# BÀI THỰC HÀNH 3: HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH WORD2VEC TIẾNG VIỆT
- **Môn học**: Xử lý Ngôn ngữ Tự nhiên
- **Mục tiêu**:
  1. Xây dựng pipeline đọc dữ liệu theo luồng (streaming) cho tập dữ liệu lớn.
  2. Huấn luyện mô hình Word2Vec (Skip-gram & Negative Sampling) bằng thư viện Gensim.
  3. Đánh giá tính tương đồng ngữ nghĩa (Word Similarity), phép toán tương tự (Word Analogy).
  4. Trực quan hóa không gian nhúng từ bằng t-SNE 2D.

In [1]:
import re
import os
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

### BƯỚC 1: XÂY DỰNG DATA PIPELINE CHO TIẾNG VIỆT
- Dữ liệu trong `datatrain.txt` là dữ liệu tiếng Việt được trích xuất từ Wikipedia/tin tức, đã được tách từ ghép sẵn bằng dấu gạch dưới `_` (như: `đơn_giản`, `lãnh_đạo`, `tổng_thống`).
- Do đó, ta không cần gọi lại mô hình CRF tokenizer (`underthesea.word_tokenize`) trên từng dòng nữa. Việc tokenize lại vừa gây sai lệch từ ghép vừa làm chậm hệ thống hàng nghìn lần.
- Sử dụng Generator `VietnameseCorpus` để stream từng dòng, chuẩn hóa chữ thường và phân tách bằng `split()`.

In [2]:
class VietnameseCorpus:
    def __init__(self, corpus_path, max_lines=None):
        self.corpus_path = corpus_path
        self.max_lines = max_lines

    def __iter__(self):
        with open(self.corpus_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if self.max_lines and i >= self.max_lines:
                    break
                line = line.lower()
                # Giữ nguyên các chữ cái, chữ số và dấu gạch dưới của từ ghép tiếng Việt
                line = re.sub(r'[^\w\s]', ' ', line)
                tokens = [t for t in line.strip().split() if len(t) > 1]
                if tokens:
                    yield tokens

CORPUS_PATH = "datatrain.txt"
print(f"Đường dẫn tập dữ liệu: {CORPUS_PATH}")

### BƯỚC 2: HUẤN LUYỆN MÔ HÌNH WORD2VEC (SKIP-GRAM)
- `vector_size=100`: Chiều dài không gian biểu diễn nhúng.
- `window=5`: Cửa sổ ngữ cảnh trước và sau từ mục tiêu.
- `min_count=10`: Tần suất tối thiểu của từ để đưa vào từ điển (loại bỏ từ hiếm, nhiễu).
- `sg=1`: Sử dụng kiến trúc Skip-gram.
- `negative=5`: Kỹ thuật Negative Sampling với k=5 mẫu âm.

In [3]:
# Sử dụng 500,000 dòng để tối ưu tốc độ huấn luyện trên máy cá nhân
TRAIN_LINES = 500000
sentences = VietnameseCorpus(CORPUS_PATH, max_lines=TRAIN_LINES)

print(f"Bắt đầu huấn luyện mô hình Word2Vec (Skip-gram, max {TRAIN_LINES:,} dòng)...")
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=10,
    workers=4,
    sg=1,
    negative=5,
    epochs=5
)

# Lưu mô hình
MODEL_NAME = "word2vec_skipgram_vi.model"
model.save(MODEL_NAME)
print(f"Huấn luyện thành công và lưu mô hình tại: {MODEL_NAME}")
print(f"Tổng số từ trong từ điển (Vocabulary size): {len(model.wv):,}")

### BƯỚC 3: TRUY VẤN TỪ ĐỒNG NGHĨA (WORD SIMILARITY)

In [4]:
query_words = ['tổng_thống', 'chiến_tranh', 'kinh_tế', 'giáo_dục', 'việt_nam']

print("=== TÌM KIẾM CÁC TỪ TƯƠNG ĐỒNG NHẤT (TOP 5 SIMILAR WORDS) ===")
for word in query_words:
    if word in model.wv:
        similar_words = model.wv.most_similar(word, topn=5)
        print(f"\nTừ khóa: [{word}]")
        for sim_word, score in similar_words:
            print(f"  - {sim_word:<20} | Similarity: {score:.4f}")
    else:
        print(f"Từ '{word}' không có trong từ điển.")

### BƯỚC 4: KIỂM THỬ QUAN HỆ TƯƠNG TỰ NGỮ NGHĨA (WORD ANALOGY)
- Phép toán vector kinh điển: $\vec{w} = \vec{w}_{pos1} - \vec{w}_{neg1} + \vec{w}_{pos2}$

In [5]:
def evaluate_analogy(pos1, neg1, pos2):
    try:
        results = model.wv.most_similar(positive=[pos1, pos2], negative=[neg1], topn=3)
        print(f"Phép toán: '{pos1}' - '{neg1}' + '{pos2}' =")
        for word, score in results:
            print(f"  -> {word:<20} | Cosine: {score:.4f}")
    except KeyError as e:
        print(f"Từ không có trong vocab: {e}")

print("=== ĐÁNH GIÁ WORD ANALOGY ===")
evaluate_analogy('hà_nội', 'việt_nam', 'pháp')
print()
evaluate_analogy('tổng_thống', 'hoa_kỳ', 'nga')

### BƯỚC 5: TÍNH ĐỘ TƯƠNG ĐỒNG COSINE GIỮA CÁC CẶP TỪ

In [6]:
pairs = [
    ('thành_phố', 'đô_thị'),
    ('chiến_tranh', 'xung_đột'),
    ('học_sinh', 'sinh_viên'),
    ('kinh_tế', 'thời_tiết'),
    ('quân_đội', 'nấu_ăn')
]

print("=== COSINE SIMILARITY GIỮA CÁC CẶP TỪ ===")
for w1, w2 in pairs:
    if w1 in model.wv and w2 in model.wv:
        sim = model.wv.similarity(w1, w2)
        print(f"Similarity('{w1}', '{w2}') = {sim:.4f}")

### BƯỚC 6: TRỰC QUAN HÓA KHÔNG GIAN VECTOR BẰNG T-SNE 2D

In [7]:
sample_words = [
    'việt_nam', 'pháp', 'mỹ', 'anh', 'nga', 'trung_quốc', 'nhật_bản',
    'hà_nội', 'paris', 'luân_đôn', 'bắc_kinh', 'tokyo', 'moskva',
    'tổng_thống', 'thủ_tướng', 'đại_tá', 'lãnh_đạo', 'chính_trị',
    'chiến_tranh', 'quân_đội', 'vũ_khí', 'xung_đột', 'hòa_bình',
    'kinh_tế', 'thương_mại', 'tài_chính', 'ngân_hàng', 'đầu_tư',
    'giáo_dục', 'trường_học', 'học_sinh', 'sinh_viên', 'giáo_viên'
]

valid_words = [w for w in sample_words if w in model.wv]
word_vectors = np.array([model.wv[w] for w in valid_words])

tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(valid_words)-1))
vectors_2d = tsne.fit_transform(word_vectors)

plt.figure(figsize=(14, 10))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], c='crimson', edgecolors='k', s=90, alpha=0.8)

for i, word in enumerate(valid_words):
    plt.annotate(
        word,
        xy=(vectors_2d[i, 0], vectors_2d[i, 1]),
        xytext=(6, 3),
        textcoords='offset points',
        fontsize=11,
        fontweight='medium'
    )

plt.title('Phân bố không gian ngữ nghĩa Word2Vec Tiếng Việt (t-SNE 2D)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('word2vec_tsne.png', dpi=300)
plt.show()
print('Đã lưu biểu đồ trực quan hóa: word2vec_tsne.png')

### BƯỚC 7: KẾT LUẬN & SO SÁNH SKIP-GRAM VS CBOW
1. **Hiệu quả của tiền xử lý**: Giữ nguyên dấu gạch dưới `_` giúp bảo tồn từ ghép tiếng Việt trong từ vựng, tránh phân mảnh âm tiết.
2. **Skip-gram**: Học rất tốt ngữ nghĩa của các từ hiếm và quan hệ ngữ cảnh rộng trong văn bản chính trị/lịch sử.
3. **Word Analogy**: Phép cộng trừ vector thể hiện quan hệ địa lý và thể chế tương đối nhất quán.